In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → 다음 셀부터
# (google-colab·numba·cudf 버전 경고와 bottleneck 경고는 무시해도 된다)
# 로컬은 .venv에 requirements.txt로 이미 설치되어 있으면 건너뛴다: python -m pip install -r requirements.txt
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 런타임 → 세션 다시 시작 후 다음 셀부터 실행')
else:
    print('로컬: 설치 생략 (requirements.txt 버전 사용)')

# model_test — H2 편입: 선형 + LightGBM (로컬 CPU)

H2(팀원 구현, `src/h2/`)를 v2 모델 비교에 넣는다. 정의·결정은 `config/experiment.yaml`의 `H2` 칸, 계산 코드는 `src/features/h2.py`·`src/evaluation/model_test.py`. 이 노트북은 함수만 부른다.

- **H2 정의(팀원 그대로)**: `debt` = 부채비율(결산+6개월 공개 가정, 12월 결산만)을 매달 적격 종목 안 1%/99% 윈저 → z(ddof=0), `policy` = 한국은행 통화정책방향 점수(`final_rule_v2_sensitivity`), `policy_x_debt` = policy × debt.
- **순위 변환 예외**: `debt`·`policy_x_debt`는 매달 순위 [−1, 1]로 바꾸지 않고 z 그대로 넣는다(순위로 바꾸면 정책 세기가 사라짐). 예측 때 빈 값은 0.
- **입력 조합**: 개별형 [Size, B/M, Mom, σ, debt, policy×debt] + 비교용 [Size, B/M, Mom, σ] / 지수형 [H1 지수, σ, H2 지수] + 비교용 [H1 지수, σ]. 비교용도 H2 조합과 **같은 학습 행**.
- **학습 행**: H1·σ·목표값·H2가 모두 있는 행, 목표연도 **2004년부터**(부채 값이 2002-07부터라 2003년 H2 지수를 만들 수 없음). Fold 계획은 그대로(Fold 1은 2004년만으로 학습). 채점 행은 그대로 133,876행.
- **지수 ①**: H1 지수는 기존 실험과 같은 행으로 만들어 기존과 같은 숫자다. H2 지수는 [debt, policy×debt]로 같은 ① 규칙(해마다 Y−2까지 학습, Y−1로 고름), 순위로 바꾸지 않고 ① 예측 − ①(입력 0) 예측.
- **입력**: `data/`와 config만(이전 실행 기록을 읽지 않음). 처음 실행이면 2번 셀이 `data/raw/h2/`의 CSV로 `data/processed/h2.parquet`를 만든다.
- **이어서 돌리기 없음**: 셀마다 계산 직후 검사·저장한다. 세션이 끊기면 0~3번을 다시 돌리고 저장 안 된 셀부터.
- 결과 폴더: `outputs/model_results/<AUTHOR>/<시각>_h2_<모델·단계>/` (Colab은 내 드라이브 `model_results/<AUTHOR>/`).
- **이 노트북**: 로컬 CPU(.venv 커널)에서 선형(4번)·LightGBM(6번). XGBoost는 `model_test_h2_xgb.ipynb`(Colab GPU).

## 0. 실행 환경 확인

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil
from pathlib import Path

print('python      :', sys.version.split()[0], platform.platform())
print('작업 폴더    :', os.getcwd())
print('/content 존재:', Path('/content').exists())

# 저장소가 보이는지: 위로 올라가며 src/와 config/experiment.yaml을 찾는다
found = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file()), None)
print('저장소 루트   :', found)

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('GPU          : 없음')

for m in ['numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn', 'xgboost', 'lightgbm', 'yaml']:
    if importlib.util.find_spec(m):
        mod = __import__(m)
        print(f'{m:9s}', getattr(mod, '__version__', '?'))
    else:
        print(f'{m:9s}', '설치 안 됨')

## 1. 경로·설정

- **로컬**: 선형·LightGBM. `WORKERS`는 선형 Fold 병렬 CPU 수(로컬 8개 중 6개).
- **Colab**(이 노트북은 보통 로컬에서 돌림, Colab CPU로 돌릴 때만): `GIT_REF`에 **브랜치 이름**을 적는다(그 브랜치의 GitHub 최신 코드를 받음 → 로컬 수정은 푸시 후). 런타임 유형 GPU → 아래 설치 줄 실행 → 런타임 다시 시작 → 0번부터.
  `!pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3`
- Colab의 `data/`는 내 드라이브 `금융데이터인텔리전스/data`. **`data/raw/h2/`(CSV 5개)가 그 안에 있어야 한다.**

In [ ]:
import os, sys, re, shutil, subprocess, time
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

AUTHOR = ''              # 결과 폴더 outputs/model_results/<작성자>/ (영문·숫자·_·-만)
EXPERIMENT = 'h2'        # 결과 폴더 이름 <시각>_<실험>_<모델·단계>
GIT_REF = ''             # Colab에서만: 브랜치 이름(예: 'jiun', 'main') — 그 브랜치의 GitHub 최신 코드. 특정 시점 재현은 커밋 해시

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')
if not AUTHOR:
    raise ValueError('AUTHOR를 채운다(결과 폴더 outputs/model_results/<AUTHOR>/).')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름(예: 'jiun', 'main')을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    # GitHub에서 최신을 받아 GIT_REF로 맞춘다: 브랜치 이름이면 원격 최신(origin/<브랜치>), 아니면 커밋 해시
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소의 data/(.gitkeep만 있음)를 Drive 공유 data로 바꿔 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # Colab: 내 드라이브(개인) → 내려받아 로컬 outputs/model_results/에 넣고 커밋
    WORKERS = 1
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None    # 기본: ROOT/outputs/model_results (<작성자>/<결과 폴더>)
    WORKERS = 6

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.evaluation import model_test as mt
from src.features import h2 as h2_features
from src.models import boosting

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
pins = dict(re.findall(r'^([A-Za-z0-9_.-]+)==([^\s#]+)', (ROOT / 'requirements.txt').read_text(encoding='utf-8'), re.M))
for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'pyarrow', 'PyYAML', 'xgboost', 'lightgbm']:
    try:
        got = version(name)
    except PackageNotFoundError:
        got = '없음'
    print(f"{name:13s} 고정 {pins.get(name, '?'):8s} 설치 {got:8s} {'' if got == pins.get(name) else '← 다름'}")
GPU_NAME = (subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
            if shutil.which('nvidia-smi') else None)
N_THREADS = os.cpu_count()
started = datetime.now()
pd.set_option('display.width', 200, 'display.max_rows', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT   :', ROOT)
print('저장   :', (OUT_ROOT or ROOT / config['PATHS']['model_results']) / AUTHOR)
print('GPU    :', GPU_NAME, '| 스레드:', N_THREADS, '| WORKERS:', WORKERS, '| 시작:', started.isoformat(timespec='seconds'))

## 2. H2 특징표 (약 10초)

`data/raw/h2/`의 팀원 CSV 2개(부채 6개월 지연, 월별 정책 점수)와 우리 패널(적격)로 `data/processed/h2.parquet`를 만든다. 파일이 이미 있으면 다시 계산한 값과 **완전히 같은지**만 확인한다(다르면 멈춤).
검사: 열·중복, 6개월 지연·공개일 ≤ 기준월 말·12월 결산, 정책 점수 월당 1개·발표일 < 월말, 기준월 2021-12(테스트 기간) 행 버림. 확인할 숫자: 174,260행, 부채 있는 행 157,836(90.6%), 부채 값 중 적격 행에 붙은 비율 100%, 정책 251개월.

In [ ]:
h2_record = h2_features.ensure_h2(ROOT, config, execute=True)
pd.Series(h2_record['checks']['debt_share_by_base_year'], name='부채 있는 비율(기준연도)').to_frame().T

## 3. 입력 준비 (약 2분)

기본 입력(H1·σ·목표값)에 H2 두 열(z 그대로)을 붙인다. 확인할 값: 채점 행 **133,876**, `train_start_target_year` 2004, 연도별 학습 행(2003년은 0).

In [ ]:
t0 = time.perf_counter()
prepared = mt.prepare_inputs(ROOT, config, **mt.h2_inputs(config))
print(f'입력 준비 {time.perf_counter() - t0:.0f}초')
for k, v in prepared['checks'].items():
    print(f'  {k:45s} {v}')
SETS = mt.h2_input_sets(config)          # {'individual': {...}, 'index': {...}}
for phase, sets in SETS.items():
    print(phase, list(sets))
d = prepared['data']
print('연도별 ② 학습 행:', d[d.train].groupby('year').size().to_dict())
results, saved = {}, {}

## 4. 선형 (로컬, 약 30~40분)

OLS·Ridge × 손실 3개 × 목표값 3개 × 입력 4개(개별형 2 + 지수형 2) × Fold 16. 지수형 ①은 설정마다 H1·H2 지수를 만든다. 셀 6개 → 마지막 셀에서 합쳐 검사·저장.

In [ ]:
LINEAR_SETS = {**SETS['individual'], **SETS['index']}


def run_linear_part(model, loss):
    t = time.perf_counter()
    results[('linear', model, loss)] = mt.run_linear(prepared, config, settings=mt.part_settings(model, loss), sets=LINEAR_SETS,
                                                     workers=WORKERS)
    print(f'[{model} × {loss}] 완료 {(time.perf_counter() - t) / 60:.1f}분')
    display(mt.part_summary(results[('linear', model, loss)]))

In [ ]:
run_linear_part('ols', 'mse')

In [ ]:
run_linear_part('ols', 'huber_q999')

In [ ]:
run_linear_part('ols', 'huber_sigma135')

In [ ]:
run_linear_part('ridge', 'mse')

In [ ]:
run_linear_part('ridge', 'huber_q999')

In [ ]:
run_linear_part('ridge', 'huber_sigma135')

In [ ]:
missing = [p for p in mt.LINEAR_PARTS if ('linear', *p) not in results]
if missing:
    raise RuntimeError(f'아직 안 돌린 셀: {missing}')
merged = mt.merge_results([results[('linear', *p)] for p in mt.LINEAR_PARTS])
saved['linear'] = mt.finalize_linear(ROOT, config, prepared, merged, started=started, out_root=OUT_ROOT, author=AUTHOR,
                                     experiment=EXPERIMENT)
print(saved['linear']['run_dir'])
perf = saved['linear']['tables']['performance_summary']
show = ['model', 'target', 'loss', 'input_set', 'mean_ic', 'icir', 'ic_positive_share', 'train_minus_valid']
display(perf[perf.scope.eq('full')].sort_values('mean_ic', ascending=False)[show].head(20))

## 5. 부스팅 실행 함수

- 개별형: [Size, B/M, Mom, σ, debt, policy×debt]와 [Size, B/M, Mom, σ]. 폴더 `<시각>_h2_<종류>_individual`.
- 지수형: ①(H1·H2 지수, 설정마다 2004~2021년) → ② [H1 지수, σ, H2 지수]와 [H1 지수, σ]. 폴더 `<시각>_h2_<종류>_index`.
- 설정·조정값(깊이 1~3, 학습률 0.01, 트리 1000, span 12·36·60)은 기존 부스팅과 같다.
- XGBoost는 config상 GPU(Colab)에서만 돌린다.

In [ ]:
def run_boost(kind, phase):
    if boosting.default_device(kind) == 'cuda' and not GPU_NAME:
        raise RuntimeError('XGBoost는 GPU(Colab)로 돌린다.')
    t = time.perf_counter()
    stage1 = None
    if phase == 'index':
        stage1 = mt.run_boosting_stage1(prepared, config, kind, n_threads=N_THREADS)
        print(f'지수형 ① {(time.perf_counter() - t) / 60:.1f}분')
    key = (kind, phase)
    results[key] = mt.run_boosting(prepared, config, kind, phase=phase, stage1=stage1, sets=SETS[phase], n_threads=N_THREADS)
    print(f'{kind} {phase} 계산 {(time.perf_counter() - t) / 60:.1f}분')
    saved[key] = mt.finalize_boosting(ROOT, config, prepared, results[key], started=started, out_root=OUT_ROOT,
                                      author=AUTHOR, experiment=EXPERIMENT)
    print(saved[key]['run_dir'])
    return mt.boosting_part_summary(results[key])

## 6. LightGBM (로컬 CPU)

예상 시간은 기존 실측(H2 없음)에서 입력 수에 비례해 어림한 값이다(개별형은 입력 조합 2개라 약 2배, 지수형은 ①이 H1·H2 두 개).

In [ ]:
# 6-1. LGB 순위 회귀 — 개별형 (예상 약 40분)
run_boost('lgb_rankreg', 'individual')

In [ ]:
# 6-2. LGB 순위 회귀 — 지수형 (예상 약 1~1.5시간)
run_boost('lgb_rankreg', 'index')

In [ ]:
# 6-3. LGB 회귀 — 개별형 (예상 약 2시간)
run_boost('lgb_reg', 'individual')

In [ ]:
# 6-4. LGB 회귀 — 지수형 (예상 약 4~5시간)
run_boost('lgb_reg', 'index')